# Demo + Deploy (model already on HF — no training)

Video **money shot** + deploy a **free CPU Hugging Face Space**. You can deploy
either from your LoRA **adapter** or from a **merged** model (recommended for a
CPU Space: simpler + faster to load). Run top to bottom. Needs an HF token.

## 1. Install + get the repo helpers

In [ ]:
!pip install -q transformers torch gradio huggingface_hub peft
import os
if not os.path.isdir("QuestionGen"):
    !git clone -q https://github.com/gabriel-xiong/QuestionGen.git
%cd QuestionGen

## 2. Config — SET THESE

In [ ]:
HF_USER    = "gabriel-xiong"
ADAPTER_ID = "gabriel-xiong/apbio-item-generator-qwen3-1.7b-lora"   # your LoRA adapter
MERGED_ID  = "gabriel-xiong/apbio-item-generator-qwen3-1.7b"        # merged repo to create
BASE_ID    = "Qwen/Qwen3-1.7B"
SPACE_ID   = f"{HF_USER}/apbio-item-generator"
USE_MERGED = True   # True = Space uses the merged model (run step 5 first); False = use adapter

## 3. Log in to Hugging Face

In [ ]:
from huggingface_hub import login
login()  # paste your hf_... WRITE token

## 4. Video money shot — base vs tuned on the SAME (unseen) prompt

In [ ]:
import json, sys, torch; sys.path.insert(0,'scripts')
import gen_spec
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def gen(m, tok, msgs):
    x = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    i = tok(x, return_tensors='pt').to(m.device)
    o = m.generate(**i, max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][i['input_ids'].shape[1]:], skip_special_tokens=True)

sc = json.loads(open('data/eval_scenarios_ood.jsonl').readline())   # unseen topic
system, user = gen_spec.build_generation_prompt(sc['topic'], sc['misconception_ids'])
msgs = [{'role':'system','content':system},{'role':'user','content':user}]

tok = AutoTokenizer.from_pretrained(BASE_ID)
base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype='auto', device_map='auto').eval()
print('=== BASE Qwen3-1.7B ===
', gen(base, tok, msgs))
tuned = PeftModel.from_pretrained(base, ADAPTER_ID).eval()
print('
=== TUNED (yours) ===
', gen(tuned, tok, msgs))

## 5. Build + push a MERGED model (recommended for the CPU Space)
Merges your adapter into the base and pushes an fp16 model the Space can load
with plain transformers. Skip if you set `USE_MERGED = False`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
if USE_MERGED:
    base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype='float16', device_map='auto')
    merged = PeftModel.from_pretrained(base, ADAPTER_ID).merge_and_unload()
    mtok = AutoTokenizer.from_pretrained(BASE_ID)
    merged.push_to_hub(MERGED_ID)
    mtok.push_to_hub(MERGED_ID)
    print('pushed merged model ->', MERGED_ID)
else:
    print('skipped (USE_MERGED=False)')

## 6. Deploy the free CPU Space
Points the app at the merged model (or the adapter, if `USE_MERGED=False`).

In [ ]:
import re
target = MERGED_ID if USE_MERGED else ADAPTER_ID
src = open('scripts/app.py', encoding='utf-8').read()
src = re.sub(r'MODEL_ID = "[^"]*"', f'MODEL_ID = "{target}"', src, count=1)
src = re.sub(r'ADAPTER = (True|False)', f'ADAPTER = {not USE_MERGED}', src, count=1)
open('app_space.py','w',encoding='utf-8').write(src)
open('requirements.txt','w').write('transformers
torch
gradio
peft
')

from huggingface_hub import HfApi, create_repo
create_repo(SPACE_ID, repo_type='space', space_sdk='gradio', exist_ok=True)
api = HfApi()
api.upload_file(path_or_fileobj='app_space.py', path_in_repo='app.py', repo_id=SPACE_ID, repo_type='space')
api.upload_file(path_or_fileobj='requirements.txt', path_in_repo='requirements.txt', repo_id=SPACE_ID, repo_type='space')
api.upload_file(path_or_fileobj='data/apbio_misconceptions.json', path_in_repo='data/apbio_misconceptions.json', repo_id=SPACE_ID, repo_type='space')
print('Space building: https://huggingface.co/spaces/' + SPACE_ID)
print('using', 'MERGED '+MERGED_ID if USE_MERGED else 'ADAPTER '+ADAPTER_ID)

## Done
Permanent free-CPU Space URL. Merged model loads faster than base+adapter on CPU.
For GPU speed, switch the Space hardware to ZeroGPU/GPU in Settings (no code change).